# Phase 3 — Hybrid Search + Evaluation

Goal: make retrieval better, then measure how much better. Phase 2 left `hybrid_search` (pgvector + Postgres full-text, fused with RRF) as the retrieval baseline — this phase adds a cross-encoder reranker on top, then (next) RAGAS + LLM-as-judge evaluation comparing baseline vs hybrid vs hybrid+rerank.

This notebook starts with the reranker — the first open item in[docs/phase-2-rag-pipeline.md](../docs/phase-2-rag-pipeline.md)'s Phase 3 backlog.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "..")

## Cross-encoder reranker

`hybrid_search` returns its top-k by RRF-fused rank — a cheap combination of two independent scores (lexical + vector), not a model that actually reads the query and the chunk together. A cross-encoder does: it scores each (query, chunk) pair jointly, which is slower per pair but more accurate at judging relevance. Standard pattern: retrieve a wider top-k cheaply, rerank down to a narrower top-k with the cross-encoder.

`src/rerank.py` adds this: `rerank(query, chunks, top_k=3)`, using `cross-encoder/ms-marco-MiniLM-L-6-v2` (`sentence-transformers`, already a dependency — same library as the embedding model, no new package).

In [2]:
from src.db import get_conn, hybrid_search
from src.embeddings import embed_texts
from src.rerank import rerank

def retrieve(query: str, top_k: int = 5) -> list[dict]:
    embedding = embed_texts([query])[0]
    with get_conn() as conn:
        return hybrid_search(conn, query, embedding, top_k=top_k)

### Case 1 — a straightforward SEC-filing claim

`hybrid_search` top-5, before reranking:

In [3]:
claim = "Apple's revenue for fiscal year ending 2025-09-27 was $416,161,000,000."

top5 = retrieve(claim, top_k=5)
for r in top5:
    print(f"{r['score']:.4f}  [{r['source']}] {r['title']}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

0.0328  [secedgar] Apple Inc. (AAPL) — Revenue
0.0315  [secedgar] Apple Inc. (AAPL) — Revenue
0.0315  [secedgar] Apple Inc. (AAPL) — Revenue
0.0300  [secedgar] Apple Inc. (AAPL) — Total assets
0.0292  [secedgar] Apple Inc. (AAPL) — Revenue


Reranked down to top-3:

In [4]:
top3 = rerank(claim, top5, top_k=3)
for r in top3:
    print(f"{r['rerank_score']:.4f}  [{r['source']}] {r['title']}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

10.2322  [secedgar] Apple Inc. (AAPL) — Revenue
8.2994  [secedgar] Apple Inc. (AAPL) — Revenue
8.2237  [secedgar] Apple Inc. (AAPL) — Revenue


### Case 2 — a fuzzier, definitional claim

Case 1 is easy — the entity/metric keywords in the claim already overlap with the right chunk, so RRF alone likely gets it right. A Wikipedia-sourced definitional claim is a better test: no numeric anchor, more chunks compete on topic similarity alone.

In [5]:
claim2 = (
    "The price-to-earnings ratio is calculated by dividing a company's "
    "market capitalization by its total revenue."
)

top5_2 = retrieve(claim2, top_k=5)
for r in top5_2:
    print(f"{r['score']:.4f}  [{r['source']}] {r['title']}")

0.0328  [wikipedia] Price–earnings ratio
0.0315  [wikipedia] Price–earnings ratio
0.0304  [wikipedia] Price–earnings ratio
0.0289  [wikipedia] Price–earnings ratio
0.0287  [wikipedia] Price–earnings ratio


In [6]:
top3_2 = rerank(claim2, top5_2, top_k=3)
for r in top3_2:
    print(f"{r['rerank_score']:.4f}  [{r['source']}] {r['title']}")

5.3848  [wikipedia] Price–earnings ratio
4.6426  [wikipedia] Price–earnings ratio
1.5628  [wikipedia] Price–earnings ratio


## Measuring it: `eval/compare_retrieval.py`

The two cases above look right by eye, but "looks right" isn't a measurement.
`eval/compare_retrieval.py` scores every retrieval strategy against
`data/eval_claims.csv` on two metrics:

- **hit_rate@k** — out of all claims, what fraction had the correct document
  somewhere in the top-k results? (`hits / total`, e.g. 63/68 = 93%)
- **MRR@k** (Mean Reciprocal Rank) — same claims, but scored by *how high*
  the correct document ranked, not just whether it showed up. Each claim
  scores `1/rank` (1st place = 1.0, 2nd = 0.5, 3rd = 0.33...), averaged
  across all claims. Two strategies can tie on hit_rate but differ on MRR —
  hit_rate says "found it", MRR says "found it near the top or buried at
  the bottom."

In [7]:
import subprocess
print(subprocess.run(["uv", "run", "python", "-m", "eval.compare_retrieval"], cwd="..", capture_output=True, text=True).stdout)

evaluating on 68 claims (INSUFFICIENT rows excluded)

    minsearch: hit_rate@5=58/68 (85%)   MRR@5=0.853
      pg_text: hit_rate@5=53/68 (78%)   MRR@5=0.663
    pg_vector: hit_rate@5=55/68 (81%)   MRR@5=0.809
   hybrid_rrf: hit_rate@5=63/68 (93%)   MRR@5=0.873
hybrid_rerank: hit_rate@3=63/68 (93%)   MRR@3=0.919



## Eval dataset expansion — the first run was too easy

The eval set started at 52 claims, almost all `secedgar`/`worldbank`: 0 claims
against `fred` (5 docs, 3332 chunks) and 1 against `wikipedia` (29 docs, 1896
chunks). Hybrid search scored a perfect 100% hit_rate / 1.000 MRR on that
set — a ceiling that hides whatever the reranker actually contributes, not
evidence it doesn't matter.

Added 24 claims (`data/eval_claims.csv`, ids 53-76): 12 FRED (GDP, unemployment,
Fed funds rate, 10-year Treasury yield, CPI — real values pulled from the DB,
not invented) and 12 Wikipedia (definitional claims from articles already
indexed). Also added a `fred` branch to `is_relevant()` in
`compare_retrieval.py` — it had no case for that source, so every FRED claim
would've silently scored as a miss regardless of retrieval quality.

The numbers above are already the updated ones (76 claims, 68 non-INSUFFICIENT):
`hybrid_rrf` hit_rate@5=93% MRR@5=0.873 -> `hybrid_rerank` hit_rate@3=93%
MRR@3=0.919. No longer a ceiling — reranking still improves ranking quality.

### Why the misses happen — a source collision, not a vocabulary problem

All 5 remaining misses are FRED claims. Here's what `hybrid_search` actually
retrieves for one of them:

In [8]:
miss_claim = "US GDP in the third quarter of 2019 was about $21.7 trillion."

for r in retrieve(miss_claim, top_k=5):
    print(f"{r['score']:.4f}  [{r['source']}] {r['title']}")

0.0283  [worldbank] GDP (current US$) — Germany
0.0164  [wikipedia] Recession
0.0164  [worldbank] GDP (current US$) — United States
0.0161  [wikipedia] Hedge fund
0.0161  [worldbank] GDP (current US$) — United States


Every result is World Bank's "GDP (current US\$) — United States" or a
generic Wikipedia page — never FRED's `GDP` series chunk. Same underlying
concept, different source, close enough lexically that hybrid search picks
the wrong one. This is concrete evidence for query rewriting (rewrite the
claim to make the source/granularity explicit — "quarterly FRED GDP
reading" vs. an annual World Bank figure) — a real problem to solve, not a
rubric checkbox with nothing behind it.

## RAGAS + LLM-as-judge evaluation

Hit_rate/MRR only measure retrieval. They don't check whether the judge's
**verdict** actually follows from the evidence it was given, or whether it
quietly used its own background knowledge instead. That's what
[RAGAS](https://github.com/explodinggradients/ragas) adds — the same
LLM-as-judge idea `verifier.py` already uses, but with standardized,
pre-built prompts for specific questions:

- **Faithfulness** — does every claim in the judge's answer trace back to
  the retrieved evidence, or did it add something the evidence doesn't say?
- **Context precision** — was the retrieved evidence actually relevant to
  the question, or was it noise the judge had to ignore?

Both come from `ragas.metrics.collections` — this is `ragas==0.4.3`, whose
public API is a set of typed classes here, not the `from ragas.metrics import
faithfulness` style shown in most tutorials (that's an older version's
interface).

In [9]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecisionWithoutReference, Faithfulness

from eval.ragas_eval import ragas_llm

llm = ragas_llm()
faithfulness = Faithfulness(llm=llm)
context_precision = ContextPrecisionWithoutReference(llm=llm)

Score the Apple revenue case from above — the judge's actual verdict, against
the actual reranked evidence it saw:

In [10]:
from src.verifier import verify_claim
from src.claim_extractor import Claim

apple_claim = Claim(text=claim, entity="Apple", metric="Revenue", value=416161000000.0, date="2025-09-27")
verdict = verify_claim(apple_claim)
evidence = [c["content"] for c in top3]
response_text = f"{verdict.verdict}: {verdict.quote or verdict.reasoning}"

f = await faithfulness.ascore(user_input=claim, response=response_text, retrieved_contexts=evidence)
cp = await context_precision.ascore(user_input=claim, response=response_text, retrieved_contexts=evidence)
print("faithfulness:", f.value)
print("context_precision:", cp.value)

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01kv89fmcwerxvx632e99f7dvx` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199573, Requested 631. Please try again in 1m28.128s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

### Two prompts, not one

The rubric asks for "LLM evaluation, 2 prompts" — `verifier.py` now has
`VERDICT_PROMPT_V1` (original) and `VERDICT_PROMPT_V2` (asks for step-by-step
`reasoning` *before* deciding — the same pattern the
[LLM Zoomcamp agent-evaluation lesson](https://github.com/DataTalksClub/llm-zoomcamp/blob/main/04-evaluation/lessons/14-agent-evaluation.md)
uses: a `reasoning` field before the score in the judge's own output schema).
`Verdict.reasoning` is now a required field either way, so both prompts can
be compared on the same schema.

`eval/ragas_eval.py` runs both prompts across a sample of `eval_claims.csv`,
scoring verdict accuracy (our own metric, vs. `expected_verdict`) plus
faithfulness/context precision (RAGAS) for each:

```bash
uv run python -m eval.ragas_eval
```

**A real free-tier constraint, found running this:** OpenRouter's daily free
quota (50 requests) ran out partway through testing. Groq's `openai/gpt-oss-20b`
is what actually produced the numbers below — its own free tier caps at 8000
tokens/minute, tight enough that back-to-back calls for one claim's scoring
(judge + faithfulness + context precision) can trip a transient 429; that's
why `eval/ragas_eval.py`'s client sets `max_retries=5` — enough for
`instructor`'s own backoff to absorb it, not a proactive retry loop we wrote
ourselves.

**Results (sample of 10, seed=42) — partial on two providers:**

| Prompt | Provider | Accuracy | Faithfulness | Context precision |
|---|---|---|---|---|
| `VERDICT_PROMPT_V1` | Groq `openai/gpt-oss-20b` | 90% (9/10) | 0.905 | 0.929 |
| `VERDICT_PROMPT_V1` | OpenRouter `poolside/laguna-xs-2.1:free` | 100% (5/5 before quota hit) | 0.792 | 1.000 |
| `VERDICT_PROMPT_V2` | both | incomplete | — | — |

**What actually happened running this, on two separate providers:** free-tier
daily quotas, not code bugs, blocked V2 both times.

- Groq caps `gpt-oss-20b` at 8000 tokens/minute *and* 200,000/day. V1 alone
  (10 claims x 3 calls each) burned 198,191/200,000 — V2 got blocked almost
  immediately after.
- A fresh OpenRouter account (new API key, hoping for a clean daily quota)
  hit the *same* wall mid-way through V1 — turns out the free tier's 50
  req/day reset is a fixed daily boundary, not a per-account clock, so a new
  account doesn't buy a full reset.

Back-of-envelope: 76 claims x 2 prompts x 3 calls ≈ 456 requests — past
what either free tier gives in a day. Needs a paid tier or a multi-day
run, not a code fix. Both partial `VERDICT_PROMPT_V1` runs agree directionally
(high accuracy, high faithfulness, near-perfect context precision) — not
enough data yet to call a winner between V1 and V2.

## What's next

- Query rewriting — rewrite the claim before retrieval, aimed specifically
  at the FRED-vs-World-Bank source collision found above, not a generic
  "might help" checkbox item
- Wire the winning prompt (V1 vs V2, whichever scores better) into
  `verifier.py`'s default
- Phase 4: Streamlit UI + Langfuse monitoring

[← Back to README](../README.md)